# 02 · Fine-tune (QLoRA) and evaluate

Supervised fine-tuning on the 13k training questions. The targets are built automatically from the gold `derivation` and `block_mapping`: the model learns to cite evidence blocks and write a program, and never does arithmetic itself.

**If Colab disconnects:** reconnect (Runtime → T4 GPU), then re-run the setup and training cells. Training resumes from the last checkpoint on Drive, and prediction resumes from its `.jsonl` file.

Rough T4 timing for the 3B model: ~2–3 h for 1 epoch. For a quick first run, set `MAX_EXAMPLES = 3000` (~35 min).

In [ ]:
# --- Setup: on Colab, gets the latest code + data from GitHub; also works on a local GPU ---
import os, sys, subprocess, shutil, random
IN_COLAB = os.path.exists("/content")
if IN_COLAB:
    ROOT = "/content/finnn"
    shutil.rmtree(ROOT, ignore_errors=True)               # always fetch the latest code
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/praseeda-cse/finnn.git", ROOT], check=True)
    try:                                                  # save results on Drive so they survive disconnects
        from google.colab import drive
        drive.mount("/content/drive")
        OUT = "/content/drive/MyDrive/finsight/runs"
    except Exception as e:
        print("Drive not mounted (", e, ") - results are saved in /content/runs and are LOST if Colab disconnects")
        OUT = "/content/runs"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "peft", "bitsandbytes", "accelerate"], check=True)
else:
    ROOT = os.path.abspath("..")
    OUT = os.path.join(ROOT, "runs")
os.makedirs(OUT, exist_ok=True)
sys.path.insert(0, ROOT)

import torch
from finsight.data import load_split
from finsight.evaluate import score, print_report
from finsight.infer import load_model, predict
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - select a GPU runtime!")

docs_train, q_train = load_split(f"{ROOT}/data/tatdqa_train.json.gz")
docs_dev, q_dev = load_split(f"{ROOT}/data/tatdqa_dev.json.gz")
docs_test, q_test = load_split(f"{ROOT}/data/tatdqa_test.json.gz")
def sample(questions, n, seed=0):
    return random.Random(seed).sample(questions, n) if n < len(questions) else questions
def latest_adapter(run):
    """The finished adapter, or the newest checkpoint if training stopped midway."""
    import glob
    if os.path.exists(f"{run}/final"):
        return f"{run}/final"
    ckpts = sorted(glob.glob(f"{run}/checkpoint-*"), key=lambda p: int(p.rsplit("-", 1)[1]))
    assert ckpts, f"No adapter in {run} - run notebook 02 (training) first"
    print("NOTE: training has not finished; using", ckpts[-1])
    return ckpts[-1]
print(len(q_train), len(q_dev), len(q_test), "questions")

In [ ]:
from finsight.train import train

MODEL_ID = "Qwen/Qwen2.5-3B-Instruct"   # T4-friendly. With an A100/L4, try "Qwen/Qwen2.5-7B-Instruct"
MAX_EXAMPLES = None                     # e.g. 3000 for a quick first run
RUN = f"{OUT}/sft_{MODEL_ID.split('/')[-1]}" + (f"_{MAX_EXAMPLES}" if MAX_EXAMPLES else "")

adapter = train(MODEL_ID, docs_train, q_train, RUN, epochs=1, batch_size=1, grad_accum=16,
                max_examples=MAX_EXAMPLES, save_steps=100)

## Evaluate on dev (use dev for all tuning decisions)

In [ ]:
model, tok = load_model(MODEL_ID, adapter=f"{RUN}/final")
eval_qs = q_dev                         # or sample(q_dev, 300) for a quick check
preds = predict(model, tok, docs_dev, eval_qs, f"{RUN}/pred_dev.jsonl", batch_size=8)
result = score(eval_qs, preds)
print_report(result, "fine-tuned, dev")

In [ ]:
# Look at mistakes: this is where the next improvement comes from
rows = {r["uid"]: r for r in result["rows"]}
wrong = [q for q in eval_qs if rows[q["uid"]]["em"] == 0]
for q in wrong[:10]:
    p = preds[q["uid"]]
    print(f"[{q['answer_type']}] {q['question']}")
    print(f"   gold: {q['answer']} ({q['scale']}) | evidence {q['evidence']} | derivation {q['derivation']!r}")
    print(f"   pred: {p['answer']} ({p['scale']}) | evidence {p['evidence']}")
    print("   raw:", p["raw"].replace(chr(10), " / "), "\n")

## Final: test set. Run this only once you stop changing things
It also writes `pred_test_official.json` in the official `{uid: [answer, scale]}` format.

In [ ]:
import json
preds_test = predict(model, tok, docs_test, q_test, f"{RUN}/pred_test.jsonl", batch_size=8)
result_test = score(q_test, preds_test)
print_report(result_test, "fine-tuned, TEST")
with open(f"{RUN}/pred_test_official.json", "w") as f:
    json.dump({u: [p["answer"], p["scale"]] for u, p in preds_test.items()}, f)